# Language Processing Laboratory — Session 03
## Persian Tokenization: Space-to-Space, Word Boundaries, and Byte Pair Encoding (BPE)

**Duration:** 120 minutes  
**Level:** Undergraduate  
**Prerequisite:** Basic Python, Unicode/UTF-8 and `re` (review cells included; previous session files are not required)  
**Required libraries:** Python standard library only (`re`, `pathlib`, `collections`)

> &#x2067;**هدف جلسه:** سه راهبرد توکن‌سازی را در متن فارسی بسازید، آزمایش کنید و با رعایت تفاوت **توکنِ واژه‌ای** و **توکنِ زیرواژه‌ای** مقایسه کنید: جداسازی فاصله‌به‌فاصله، مرزبندی واژه و **Byte Pair Encoding (BPE)**.&#x2069;

**Student starter:** Complete the `TODO` code and **✍️ Exercise** explanations yourself. Public checks intentionally fail for unfinished exercises; instructor solutions are not included.

## Learning outcomes, repository, and submission

From the repository root, read `docs/AI_USE_POLICY_FA.md` and `docs/SUBMISSION_FA.md`. Do not publish completed student notebooks or answer keys in a public GitHub repository.

By the end you should be able to:

1. implement a literal **space-to-space** tokenizer and distinguish `split(' ')` from `split()`;
2. explain how Python `re` interprets the Unicode **word-boundary anchor `\b`** and why `\w` breaks a Persian word at ZWNJ;
3. write a **Persian-aware word-boundary tokenizer** with explicit rules for ZWNJ, punctuation, numbers and Latin text;
4. describe how **byte-level BPE** starts from UTF-8 bytes and repeatedly merges frequent adjacent pairs;
5. implement a single BPE merge and train/apply a tiny deterministic model **without external downloads**;
6. evaluate **word-level** tokenizers using gold token-span precision/recall/F1, and evaluate **BPE separately** by exact round trip, subword count, and bytes/token;
7. explain why direct BPE-versus-word-level F1 is **not** a fair comparison and document model limitations.

**Note:** The regex policy below is a small classroom convention, not complete Persian morphology. Byte-level BPE does **not** necessarily produce linguistic words, and individual BPE token bytes may not decode to valid UTF-8 on their own.

## Lab workflow and assessment

| Time | Activity |
|---:|---|
| 0–10 min | Environment, student metadata and annotated data |
| 10–30 min | Space-to-space tokenization and Unicode `\b` word boundaries |
| 30–60 min | Persian-aware regex word tokenizer + public checks |
| 60–85 min | UTF-8 byte-level BPE: pair merging, training and checks |
| 85–107 min | Controlled comparison: gold word spans vs BPE subwords |
| 107–120 min | Error analysis, reflection and submission |

**Rubric (10 points):** 1 pt metadata/reproducibility; 1 pt correct space/word-boundary reasoning; 2 pts Persian regex tokenizer; 2 pts BPE merge/tests; 2 pts principled comparison; 1 pt limitations/error analysis; 1 pt exit ticket + clean submission.

Save as `STUDENT_ID_session03.ipynb` and use **Restart Kernel and Run All Cells**. An unfinished student starter must **execute without exceptions**, even though its public checks show `FAIL`.

---
# 0. Environment and annotated mini-corpus

We keep ZWNJ (`U+200C`) intact. The course examples are **synthetic** and do not contain private messages. Place `data/` next to this notebook and run it from `sessions/03_tokenization/`.

In [ ]:
import sys
import re
from pathlib import Path
from collections import Counter

print('Python:', sys.version.split()[0])
print('Zero-width non-joiner (ZWNJ): U+200C; UTF-8 hex:', '\u200c'.encode('utf-8').hex())
if sys.version_info < (3, 10):
    print('WARNING: Python 3.10+ recommended.')

### Viewing Persian text accurately (نمایش صحیحِ نیم‌فاصله)

The next cell defines **output-only helpers**. They do not solve the exercises or change any Unicode characters.

- Each token appears as a numbered visual card, with Persian text isolated in a right-to-left span.
- The **actual ZWNJ** remains *inside* the word, e.g. `می‌روم`; a note outside the card indicates `U+200C`. A ZWNJ by itself is labeled explicitly.
- Literal empty tokens are shown as **∅ EMPTY STRING**, and tabs as **⟨TAB⟩**.
- The **SPACE** and **naive `\b`** algorithms are intentionally imperfect. **Gold** is the chosen annotation policy; only your completed Persian-aware implementation is meant to match it.
- Byte BPE is shown in **hexadecimal** because an individual subword piece may contain only part of a multibyte UTF-8 character.

**Note:** Python list `repr` may show escapes such as `\u200c`; our displays render the original text instead. After completing the TODO cells, use **Kernel → Restart & Run All** so your outputs are freshly generated rather than cached.

In [ ]:
# Display helpers for human-readable Persian tokens (never Python list repr).
# These helpers change only HTML presentation, not tokenization or evaluation.
from IPython.display import HTML, display
import html


def _persian_span(text):
    """Keep original code points, including U+200C, inside one isolated RTL span."""
    return ('<bdi lang="fa" dir="rtl" '
            'style="unicode-bidi:isolate;white-space:pre-wrap;letter-spacing:normal;'
            'font-family:Vazirmatn,\'Noto Naskh Arabic\',\'DejaVu Sans\',sans-serif;font-size:1.12em">'
            + html.escape(text,quote=True)+'</bdi>')


def _token_content(token):
    """Show whitespace controls explicitly, keeping joined Persian text untouched."""
    if token == '':
        return '<code dir="ltr">∅ EMPTY STRING</code>'
    if token == '\u200c':
        return '<code dir="ltr">⟨standalone ZWNJ U+200C⟩</code>'
    if token == '\t':
        return '<code dir="ltr">⟨TAB⟩</code>'
    if token == '\n':
        return '<code dir="ltr">⟨NEWLINE⟩</code>'
    if any(ch in token for ch in '\t\n\r'):
        import re as _re
        segments = _re.split(r'(\t|\n|\r)',token)
        labels = {'\t':'⟨TAB⟩','\n':'⟨NEWLINE⟩','\r':'⟨CR⟩'}
        return ''.join(('<code dir="ltr" style="padding:0 4px">'+labels[s]+'</code>')
                       if s in labels else _persian_span(s)
                       for s in segments if s)
    return _persian_span(token)


def token_cards(tokens):
    """Numbered LTR token *sequence* containing isolated RTL Persian token text."""
    if not tokens:
        return '<span dir="ltr" style="font-size:.9em">∅ (zero tokens)</span>'
    elements=[]
    for idx,token in enumerate(tokens,1):
        assert isinstance(token,str)
        zwnj_note=('<small dir="ltr" style="display:block;color:#555">contains ZWNJ U+200C</small>'
                   if '\u200c' in token and token!='\u200c' else '')
        elements.append('<span style="display:inline-flex;flex-direction:column;'
                        'align-items:center;vertical-align:top;min-height:42px;'
                        'border:1px solid #aab3bd;border-radius:7px;'
                        'padding:5px 10px;margin:3px;max-width:98%;'
                        'background:rgba(127,127,127,.055);text-align:center">'
                        f'<small dir="ltr" style="display:block;color:#666">#{idx}</small>'
                        +_token_content(token)+zwnj_note+'</span>')
    return '<div dir="ltr" style="line-height:1.75;direction:ltr;text-align:left">'+''.join(elements)+'</div>'


def show_case(text, methods, *, heading=None, note=None):
    """Show source text and each method in a legible, numbered visual table."""
    rows=[]
    for label,tokens in methods:
        rows.append('<tr><th dir="ltr" style="text-align:left;vertical-align:top;'
                    'padding:8px;white-space:nowrap;border-bottom:1px solid #ddd">'
                    +html.escape(str(label))+'</th><td style="padding:7px;'
                    'border-bottom:1px solid #ddd">'+token_cards(tokens)+'</td></tr>')
    heading_html=('<h4 dir="ltr">'+html.escape(heading)+'</h4>') if heading else ''
    notes=[]
    if '\u200c' in text:notes.append('Source contains an actual U+200C (ZWNJ), kept inside the token.')
    if '\t' in text:notes.append('A tab is marked ⟨TAB⟩ only for display; original bytes are unchanged.')
    if '  ' in text:notes.append('Multiple ASCII spaces are preserved in the source.')
    if note:notes.append(note)
    note_html=''.join('<small dir="ltr" style="display:block">'+html.escape(n)+'</small>' for n in notes)
    display(HTML('<div style="font-family:system-ui,sans-serif;margin:12px 0 18px">'
                 +heading_html+'<p dir="ltr"><b>Original:</b> '+_persian_span(text)+'</p>'
                 +note_html+'<table dir="ltr" style="border-collapse:collapse;width:100%;margin-top:8px">'
                 +''.join(rows)+'</table></div>'))


def show_word_checks(rows):
    """Display word-test results without escaped Python representations."""
    body=[]
    for num,passed,expected,actual in rows:
        label='PASS' if passed else 'FAIL'
        actual_html=token_cards(actual) if isinstance(actual,list) else html.escape(str(actual))
        body.append('<tr><th dir="ltr" style="text-align:left;vertical-align:top;padding:8px;'
                    'border-bottom:1px solid #ddd">'
                    f'{label} — case {num}</th><td style="border-bottom:1px solid #ddd;padding:6px">'
                    + '<small dir="ltr">Expected</small>'+token_cards(expected)
                    + '<small dir="ltr">Your result</small>'+actual_html+'</td></tr>')
    display(HTML('<table dir="ltr" style="border-collapse:collapse;width:100%">'
                 +''.join(body)+'</table>'))


def show_bpe_pieces(pieces, *, title='Byte-level BPE tokens'):
    """Show byte hex, not Python repr or deceptively decoded partial UTF-8."""
    cards = []
    for i, item in enumerate(pieces, 1):
        if not isinstance(item, bytes):
            raise TypeError('Each BPE token should be a bytes object.')
        try:
            label = _persian_span(item.decode('utf-8', errors='strict'))
        except UnicodeDecodeError:
            label = '<small dir="ltr">partial UTF-8 (not independently readable)</small>'
        cards.append('<span style="display:inline-flex;flex-direction:column;'
                     'vertical-align:top;align-items:center;padding:5px 8px;margin:3px;'
                     'border:1px solid #aab3bd;border-radius:7px;min-width:38px">'
                     +f'<small dir="ltr">#{i}</small><code dir="ltr">{item.hex()}</code>'
                     +label+'</span>')
    display(HTML('<div dir="ltr" style="font-family:system-ui,sans-serif;margin:10px 0">'
                 + '<b>'+html.escape(title)+'</b>'
                 + '<p><small>Hex is the exact byte value of each piece. Some pieces are not complete UTF-8 characters.</small></p>'
                 + '<div style="direction:ltr">'+''.join(cards)+'</div></div>'))


In [ ]:
# TODO: Exercise 0 — enter your own metadata.
student_id = ''
student_name = ''
university = ''
print('Student ID:', student_id)
print('Name:', student_name)
print('University:', university)

### Data and evaluation policy

The gold file is **TSV**: `id`, source `text`, and tokens delimited by `|`. Each reference token corresponds to a **word-level** policy: punctuation stands alone; an internal ZWNJ connects Persian word parts. Data are small, so report limitations rather than claiming general Persian-language superiority.

The separate `data/bpe_train.txt` contains **only training text for BPE**; no examples from the gold TSV are used to learn merges. This prevents the demonstration from fitting BPE on its evaluation samples.

In [ ]:
DATA_PATH = Path('data/tokenization_gold.tsv')
BPE_TRAIN_PATH = Path('data/bpe_train.txt')
# Each non-comment line: sample_id<TAB>text<TAB>token_1|token_2|...
gold_examples = []
for raw_line in DATA_PATH.read_text(encoding='utf-8').splitlines():
    if not raw_line or raw_line.startswith('#'):
        continue
    sample_id, text, gold = raw_line.split('\t', maxsplit=2)
    gold_examples.append((sample_id, text, gold.split('|')))

bpe_train_texts = [s for s in BPE_TRAIN_PATH.read_text(encoding='utf-8').splitlines() if s.strip()]
assert gold_examples and bpe_train_texts, 'Both local UTF-8 data files must be nonempty.'
assert len({sample_id for sample_id, _, _ in gold_examples}) == len(gold_examples), 'IDs must be unique.'
assert not ({text for _, text, _ in gold_examples} & set(bpe_train_texts)), 'No exact gold sentence may enter BPE training.'
print('Gold examples:', len(gold_examples))
print('BPE training lines:', len(bpe_train_texts))
for sample_id, text, tokens in gold_examples[:3]:
    show_case(text, [('Reference word tokens', tokens)], heading=f'{sample_id} — labeled sample')

---
# 1. Space-to-space tokenization (فاصله به فاصله)

A **literal ASCII-space** splitter uses `text.split(' ')`: it splits **only at U+0020**, and repeated spaces can produce empty strings. The more convenient `text.split()` groups multiple whitespace characters (spaces, tabs and newlines) and does not return empty pieces. Both approaches leave attached punctuation in their chunks.

For scoring, our baseline removes empty pieces from `split(' ')`. This intentionally remains a **space-to-space**, not a general whitespace, tokenizer. It does not normalize text or silently remove non-space characters.

In [ ]:
space_cases = ['من می‌روم،', 'الف  ب', 'الف\tب', 'سلام! حالت چطور است؟']
for index, text in enumerate(space_cases, 1):
    show_case(text, [
        ("split(' ') — literal ASCII space", text.split(' ')),
        ("split() — all whitespace", text.split()),
    ], heading=f'Space-splitting example {index}',
       note='These are baselines, NOT punctuation-aware word tokenizers.')

def tokenize_space_to_space(text):
    """Baseline: use only literal ASCII spaces; drop empty pieces from repeated spaces."""
    return [piece for piece in text.split(' ') if piece]

### ✍️ Exercise 1 — Diagnose the space baseline

1. Predict the outputs of `split(' ')` and `split()` on `'من  می‌روم\tبه خانه.'` **before** running an experiment.
2. Why does `split(' ')` keep an ASCII tab **inside** a token? Why is `سلام،` still one piece?
3. Does `می‌روم` contain a normal space? Verify its U+200C code point with `repr` or `ord`.

> TODO: write answers here. No hard-coded outputs from a single memorized example in your later tokenizer.

---
# 2. Word boundaries: `\b`, `\w`, and Persian-aware rules

In Python `re`, `\b` is a **zero-width boundary** between `\w` and `\W` (or at string edges). By default `\w` is Unicode-aware for alphanumeric characters and underscore, **but not ZWNJ (`U+200C`)**. Thus `re.findall(r'\b\w+\b', ...)` can split a Persian word such as `می‌روم` into pieces. The second alternative `[^\w\s]` below **keeps** punctuation and stray ZWNJ instead of silently dropping them.

Our **Persian-aware** policy treats `می‌روم` and `کتاب‌ها` as one word each, separates punctuation as individual tokens, groups Persian/Arabic/ASCII decimal digits into number tokens, and groups Latin letters separately. This is a **teaching policy**; real search engines and language models may use a different one.

In [ ]:
WORD_BOUNDARY_PATTERN = re.compile(r'\b\w+\b|[^\w\s]')

def tokenize_word_boundary(text):
    """Intentionally limited regex baseline; it does NOT protect Persian ZWNJ words."""
    return WORD_BOUNDARY_PATTERN.findall(text)

assert re.fullmatch(r'\w', '\u200c') is None
assert tokenize_word_boundary('می\u200cروم') == ['می', '\u200c', 'روم']
print('Unicode regex baseline defined. IMPORTANT: it is NOT Persian-aware.')
print('U+200C is a word character (\\w) in Python re? False')
print('See the labeled gold/baseline/Persian-aware comparison immediately after Exercise 2.')

### ✍️ Exercise 2 — Implement the Persian-aware word-boundary tokenizer

Write `tokenize_persian(text) -> list[str]` with `re.findall` or `re.finditer`, prioritizing:

1. One or more common Persian/Arabic-script **letters**, optionally followed by repeated `ZWNJ + letters` groups (ZWNJ must have a letter on **both** sides).
2. A nonempty consecutive run of `[0-9۰-۹٠-٩]`.
3. A run of `[A-Za-z]`.
4. Every other **non-whitespace** character as an individual token, including punctuation, emoji and any stray ZWNJ.

Use the starter components below. The supplied character class intentionally covers common Persian letters (not every Arabic-derived script); combining diacritics are outside this small teaching policy. Preserve original characters (no normalization, no lowercase conversion). **Do not** use `\w+` as a substitute for the Persian word rule. The code must work for unseen sentences, not only published checks.

In [ ]:
PERSIAN_LETTERS = r'[\u0621-\u063A\u0641-\u064A\u067E\u0686\u0698\u06A9\u06AF\u06CC]'
DECIMAL_DIGITS = r'[0-9۰-۹٠-٩]'
LATIN_LETTERS = r'[A-Za-z]'
ZWNJ = '\u200c'

# TODO: Exercise 2 — replace the temporary baseline with a regex tokenizer.
def tokenize_persian(text):
    """Persian-aware word tokens according to the explicit Session 03 policy."""
    return tokenize_space_to_space(text)

show_case('من می‌روم، تو می روی؟', [('Your current tokenizer (Exercise 2)', tokenize_persian('من می‌روم، تو می روی؟'))], heading='Your tokenizer — re-run after editing Exercise 2')


### Compare all three methods after completing Exercise 2

Run this cell now to study the differences, then run it again after implementing `tokenize_persian`. Each case shows the **course gold**, the two deliberately limited baselines, and **your current tokenizer**. If the TODO is unfinished, the student output can be wrong; this is expected, not a display bug. The final code should match Gold on these samples.

`#زبان_فارسی` is split into `#`, `زبان`, `_`, `فارسی` by the **specific gold policy used in this course**; other applications may keep hashtags intact.

In [ ]:
# The gold labels are given for practice; no solution implementation is provided here.
visual_cases = [
    ('من می‌روم،', ['من', 'می‌روم', '،']),
    ('کتاب‌ها!', ['کتاب‌ها', '!']),
    ('#زبان_فارسی', ['#', 'زبان', '_', 'فارسی']),
    ('سال ۱۴۰۵', ['سال', '۱۴۰۵']),
    ('سلام😊', ['سلام', '😊']),
]
for index, (text, reference) in enumerate(visual_cases, 1):
    guess = tokenize_persian(text)
    show_case(text, [
        ('GOLD — course annotation', reference),
        ('SPACE — baseline, not correct in general', tokenize_space_to_space(text)),
        ('\\b — naive regex, not ZWNJ-aware', tokenize_word_boundary(text)),
        ('YOUR Persian-aware tokenizer (Exercise 2)', guess),
    ], heading=f'Word boundary comparison {index}',
       note=f'Your current implementation matches Gold: {guess == reference}. Re-run after solving Exercise 2.')


### Public checks — Persian word boundaries

Do **not** edit these checks. Starter `TODO` code is intentionally incomplete. Both exact token lists and the requirement not to discard characters matter.

In [ ]:
def check_word_tokenization(tokenizer, *, verbose=True):
    cases = [
        ('من می‌روم، تو می روی؟', ['من', 'می‌روم', '،', 'تو', 'می', 'روی', '؟']),
        ('کتاب‌ها! دانش‌آموزان.', ['کتاب‌ها', '!', 'دانش‌آموزان', '.']),
        ('سال ۱۴۰۵ و 2026 است.', ['سال', '۱۴۰۵', 'و', '2026', 'است', '.']),
        ('Python و فارسی', ['Python', 'و', 'فارسی']),
        ('کد ۲۰۲۶-الف 😊', ['کد', '۲۰۲۶', '-', 'الف', '😊']),
        ('#زبان_فارسی', ['#', 'زبان', '_', 'فارسی']),
        ('سلام  دنیا!', ['سلام', 'دنیا', '!']),
        ('Python3 و فارسی۱۲۳', ['Python', '3', 'و', 'فارسی', '۱۲۳']),
        ('الف‌ و ‌ب', ['الف', '‌', 'و', '‌', 'ب']),
        ('', []),
    ]
    passed = 0
    table = []
    for index, (text, expected) in enumerate(cases, 1):
        try:
            got = tokenizer(text)
            ok = got == expected
        except Exception as exc:
            got, ok = f'ERROR: {exc}', False
        table.append((index, ok, expected, got))
        passed += int(ok)
    if verbose:
        show_word_checks(table)
        print(f'Persian tokenizer: {passed}/{len(cases)} checks passed')
    return passed == len(cases)

check_word_tokenization(tokenize_persian)

---
# 3. Byte Pair Encoding (BPE) — a *subword* tokenizer

**BPE does not mean splitting text into words.** A small **byte-level BPE** model begins with all **UTF-8 bytes** of each non-whitespace chunk as individual tokens, then learns a sequence of **frequent adjacent byte-pair merges** from the *training corpus only*. Each learned merge produces a new token (`bytes` object). At inference time, apply **the same merges in learned order** to new text; do **not** learn them from evaluation data.

For transparency and reproducibility, our mini implementation:

- keeps whitespace chunks as whole `bytes` tokens, preserving the original input for decoding;
- learns merges **inside non-whitespace chunks only** (never across whitespace);
- selects the most frequent adjacent pair, breaking frequency ties in lexicographic byte order;
- stops when no pair appears twice or the merge budget is reached;
- never assumes individual byte/subword tokens are valid UTF-8 strings.

**Important:** This is a deliberately simplified *byte-level BPE* exercise, not an exact reimplementation of GPT-2/tiktoken/SentencePiece. Real models may use different pre-tokenization, special tokens and vocabularies. In particular, punctuation may stay inside an initial non-whitespace chunk, so BPE pieces **need not align with reference word boundaries**.

In [ ]:
example = 'می‌روم!'
raw_bytes = example.encode('utf-8')
display(HTML('<p dir="ltr"><b>Original text:</b> '+_persian_span(example)+'</p>'))
print('Unicode code points:', [f'U+{ord(c):04X}' for c in example])
print('UTF-8 hex:', raw_bytes.hex(' '))
print('UTF-8 byte count:', len(raw_bytes))
print('Round trip:', raw_bytes.decode('utf-8') == example)
print('Do NOT decode each byte separately: Persian letters often use multiple bytes.')

### ✍️ Exercise 3 — Implement an adjacent-pair merge

`merge_adjacent_pair(symbols, pair)` accepts a list of **nonempty `bytes` objects** and the pair `(left, right)` to merge. Sweep **left-to-right** and replace each *non-overlapping* adjacent occurrence with `left + right`. Keep all other symbols unchanged.

For example, `[b'a', b'b', b'a', b'b']`, pair `(b'a', b'b')` becomes `[b'ab', b'ab']`. Overlapping occurrences must not reuse the same input symbol.

Before coding, draw the result for `[b'a', b'a', b'a']` when merging `(b'a', b'a')`. Why does the output have **two** pieces, not one? This merging step is the main coding task for BPE today.

In [ ]:
def merge_adjacent_pair(symbols, pair):
    """Merge non-overlapping occurrences of (left, right), scanning left-to-right."""
    # TODO: Exercise 3 — implement the adjacent-pair scan.
    return list(symbols)

### Public checks — BPE merge correctness

These tests cover repeated merges, overlapping candidates, nonmatches, and non-ASCII UTF-8 bytes. Do not edit them.

In [ ]:
def check_bpe_merge(merger, *, verbose=True):
    tests = [
        ([b'a', b'b', b'a', b'b'], (b'a', b'b'), [b'ab', b'ab']),
        ([b'a', b'a', b'a'], (b'a', b'a'), [b'aa', b'a']),
        ([b'c', b'a', b'b'], (b'a', b'b'), [b'c', b'ab']),
        ([b'a', b'b'], (b'b', b'a'), [b'a', b'b']),
        ([b'\xd9', b'\x85', b'!'], (b'\xd9', b'\x85'), [b'\xd9\x85', b'!']),
        ([], (b'a', b'b'), []),
    ]
    passed = 0
    for idx, (symbols, pair, expected) in enumerate(tests, 1):
        try:
            got = merger(symbols, pair)
            ok = got == expected
        except Exception as exc:
            got, ok = f'ERROR: {exc}', False
        if verbose:
            print(('PASS' if ok else 'FAIL'), f'case {idx}', [x.hex() for x in got] if isinstance(got,list) else str(got))
        passed += int(ok)
    if verbose:
        print(f'BPE merge: {passed}/{len(tests)} checks passed')
    return passed == len(tests)

bpe_merge_ready = check_bpe_merge(merge_adjacent_pair)

### BPE training and encoding — provided scaffolding

Read, step through and **explain** the code below. Only the `merge_adjacent_pair` operation was left as a `TODO`; the remaining training loop is provided to make a two-hour lab realistic. All merge statistics come from `bpe_train_texts`, never from gold evaluation texts.

`encode_byte_bpe` gives a sequence of `bytes` tokens; `b''.join(tokens).decode('utf-8')` must reproduce the input **exactly** (spaces and ZWNJ included). Do **not** individually decode each token: a byte-level subword token may end midway through a multibyte character.

In [ ]:
def byte_sequences_for_training(texts):
    """UTF-8 byte sequences for each non-whitespace chunk in training text."""
    return [[bytes([byte]) for byte in chunk.encode('utf-8')]
            for text in texts for chunk in re.findall(r'\S+', text)]


def train_byte_bpe(texts, max_merges=80, min_pair_count=2):
    """Learn deterministic adjacent byte-pair merges on training text only."""
    sequences = byte_sequences_for_training(texts)
    merges = []
    for _ in range(max_merges):
        counts = Counter(pair for seq in sequences for pair in zip(seq, seq[1:]))
        if not counts:
            break
        pair, frequency = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[0]
        if frequency < min_pair_count:
            break
        updated = [merge_adjacent_pair(seq, pair) for seq in sequences]
        if updated == sequences:
            break  # Safe starter behavior while Exercise 3 is unfinished.
        sequences = updated
        merges.append(pair)
    return merges


def encode_byte_bpe(text, merges):
    """Apply learned merges within non-whitespace chunks; preserve whitespace."""
    result = []
    for chunk in re.findall(r'\s+|\S+', text):
        if chunk.isspace():
            result.append(chunk.encode('utf-8'))
            continue
        symbols = [bytes([byte]) for byte in chunk.encode('utf-8')]
        for pair in merges:
            symbols = merge_adjacent_pair(symbols, pair)
        result.extend(symbols)
    return result

bpe_merges = train_byte_bpe(bpe_train_texts, max_merges=80)
print('Learned merges:', len(bpe_merges))
print('First 8 learned merges (hex):', [(a.hex(), b.hex()) for a,b in bpe_merges[:8]])
if not bpe_merges:
    print('Complete the merge TODO to learn actual BPE merges.')

In [ ]:
# Round-trip demonstration: BPE pieces can be incomplete UTF-8 fragments.
bpe_examples = ['من می‌روم،', 'سلام  دنیا!', 'پایتون و Python 😊', 'آزمایش\tفارسی']
for idx, text in enumerate(bpe_examples, 1):
    tokens = encode_byte_bpe(text, bpe_merges)
    restored = b''.join(tokens).decode('utf-8')
    display(HTML('<p dir="ltr"><b>BPE example '+str(idx)+':</b> '+_persian_span(text)
                 + f' | pieces: {len(tokens)} | exact byte round trip: {restored == text}</p>'))
    if idx == 1:
        show_bpe_pieces(tokens, title='BPE pieces (hex) — first example')
    assert restored == text, 'Byte BPE must never discard or change input characters.' 

### ✍️ Exercise 4 — Inspect learned BPE merges

After the BPE public checks pass:

1. Show the **first five learned byte pairs** and explain why a pair of byte values does not always represent two visible letters.
2. For `می‌روم` and `کتاب‌ها`, report initial UTF-8 byte-token counts versus learned BPE token counts.
3. Explain how BPE handles a never-seen word, and why it has no *byte-level OOV* problem on valid UTF-8 text.
4. Can BPE merge across a gold punctuation boundary such as `سلام!`? Explain why our whitespace pre-tokenization allows that and what would change with punctuation-aware pre-tokenization.

> TODO: add your computed numbers and explanations here.

---
# 4. Evaluation: three strategies, **two different units**

**Word-unit comparison:** score the **space-to-space**, naive `\b`-based, and Persian-aware regex tokenizers against the same gold **word spans**. A predicted token counts as correct only if its **exact (start, end) character offsets** equal those of one gold token. Repeated strings at different offsets are **not** interchangeable. Punctuation and ZWNJ must not be silently deleted.

**Subword-unit evaluation:** BPE tokens are `bytes` sequences that can cut across code points or word boundaries; word-level F1 is therefore **N/A**, not a low score to directly compare with other systems. Measure **lossless round trip**, **subword count**, **number of distinct byte tokens**, and **average bytes per non-whitespace BPE token** on the same gold texts. Token count here is affected by the merge budget and training data, not just a language-aware word segmentation policy.

The provided function `evaluate_tokenizer` calculates these word-span metrics; the BPE reporting function intentionally uses different statistics.
**Protocol:** Gold TSV is a small hand-annotated demonstration set, not an independent population-level benchmark. The BPE train data contain separate synthetic sentences; learned merges must never use gold TSV.


In [ ]:
def token_spans(text, tokens):
    """Align a list of *word strings*; skipped input must be whitespace only."""
    spans, cursor = [], 0
    for token in tokens:
        if not isinstance(token, str) or not token:
            raise ValueError('Each word-level token must be a nonempty string.')
        start = text.find(token, cursor)
        if start < 0 or any(not ch.isspace() for ch in text[cursor:start]):
            raise ValueError(f'Tokens lost input near index {cursor}: {tokens!r}')
        spans.append((start, start + len(token)))
        cursor = start + len(token)
    if any(not ch.isspace() for ch in text[cursor:]):
        raise ValueError('Word tokenizer discarded non-whitespace characters.')
    return spans


def evaluate_tokenizer(tokenizer, examples):
    """Micro word-span precision/recall/F1 and exact sentence count."""
    true_positive = predicted = reference = exact_sentences = 0
    errors = []
    for sample_id, text, gold in examples:
        guess = tokenizer(text)
        gold_spans = set(token_spans(text, gold))
        try:
            guess_spans = set(token_spans(text, guess))
        except (TypeError, ValueError) as exc:
            errors.append((sample_id, str(exc)))
            guess_spans = set()
        true_positive += len(guess_spans & gold_spans)
        predicted += len(guess_spans)
        reference += len(gold_spans)
        exact_sentences += int(guess == gold)
    precision = true_positive / predicted if predicted else 0.0
    recall = true_positive / reference if reference else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': round(precision, 4), 'recall': round(recall, 4),
            'f1': round(f1, 4), 'exact_sentences': exact_sentences,
            'total_sentences': len(examples), 'alignment_errors': errors}

print('Repeated-word offsets:', token_spans('سلام، سلام!', ['سلام', '،', 'سلام', '!']))

In [ ]:
word_methods = {
    'Space-to-space': tokenize_space_to_space,
    r'Naive word boundary \b': tokenize_word_boundary,
    'Persian-aware regex': tokenize_persian,
}
word_scores = {name: evaluate_tokenizer(tok, gold_examples)
               for name, tok in word_methods.items()}
print('WORD-LEVEL: exact token spans against the gold reference')
print(f"{'Method':<33} {'P':>7} {'R':>7} {'F1':>7} {'Exact':>9}")
for name, score in word_scores.items():
    print(f"{name:<33} {score['precision']:>7.3f} {score['recall']:>7.3f} "
          f"{score['f1']:>7.3f} {score['exact_sentences']:>3}/{score['total_sentences']:<5}")
print('Reminder: the Persian-aware regex TODO must be implemented before drawing conclusions.')


In [ ]:
def bpe_evaluation_summary(examples, merges):
    texts = [text for _, text, _ in examples]
    # Gather pieces *from non-whitespace chunks*. A BPE piece can be an
    # incomplete UTF-8 sequence, so never decode it to classify whitespace.
    nonspace_tokens = [piece
                       for text in texts
                       for chunk in re.findall(r'\s+|\S+', text)
                       if not chunk.isspace()
                       for piece in encode_byte_bpe(chunk, merges)]
    byte_count = sum(len(tok) for tok in nonspace_tokens)
    roundtrip_count = sum(b''.join(encode_byte_bpe(text, merges)).decode('utf-8') == text
                          for text in texts)
    return {
        'learned_merges': len(merges),
        'roundtrip_sentences': roundtrip_count,
        'total_sentences': len(texts),
        'nonwhitespace_bpe_tokens': len(nonspace_tokens),
        'distinct_bpe_byte_tokens': len(set(nonspace_tokens)),
        'avg_bytes_per_bpe_token': round(byte_count / len(nonspace_tokens), 3)
                                    if nonspace_tokens else 0.0,
    }

bpe_summary = bpe_evaluation_summary(gold_examples, bpe_merges)
print('SUBWORD-LEVEL: BPE is not word-span F1 comparable')
for key, value in bpe_summary.items():
    print(f'  {key}: {value}')
if not bpe_merges:
    print('BPE is currently the unmerged byte baseline; complete Exercise 3.')


In [ ]:
# Merge-budget experiment (learn only from BPE training text, never gold texts).
if bpe_merge_ready:
    for budget in (0, 20, 80):
        merges_at_budget = train_byte_bpe(bpe_train_texts, max_merges=budget)
        summary = bpe_evaluation_summary(gold_examples, merges_at_budget)
        print('budget=', budget,
              '| trained merges=', summary['learned_merges'],
              '| BPE tokens=', summary['nonwhitespace_bpe_tokens'],
              '| exact round trips=',
              f"{summary['roundtrip_sentences']}/{summary['total_sentences']}")
else:
    print('Complete Exercise 3 and pass the merge checks to compare budgets.')

### ✍️ Exercise 5 — Comparison and error analysis

Complete this **four-row comparison**. Use `N/A` whenever the unit is not applicable; *do not* rank word-level F1 against BPE token counts.

| Method | Unit | Word-span F1 | Exact word sentences | BPE round-trip | Non-space units |
|---|---|---|---|---|---|
| Literal space-to-space | word/chunk | TODO | TODO | N/A | N/A |
| `\b` Unicode baseline | word/punctuation | TODO | TODO | N/A | N/A |
| Persian-aware regex | word/punctuation | TODO | TODO | N/A | N/A |
| Byte-level BPE | byte/subword | N/A | N/A | TODO | TODO |

- For the first three report **word-span P/R/F1 and exact-sentence count** (same gold unit).
- For BPE report **round-trip fraction, number of non-whitespace BPE units, distinct byte-token count, and average bytes per non-whitespace unit**; write `N/A` in the word-span-F1 column and explain **why**.
- Compare outputs on **at least three sentences** including ZWNJ, punctuation and Persian/ASCII digits.
- Explain one failure of `split(' ')`, one failure of naive `\b`, and one *trade-off* of byte BPE.
- Run the provided BPE **merge-budget experiment** (0 → 20 → 80) after passing checks and interpret how the token counts change while every round trip remains exact. This is **not** the same as measured file-size compression.
- Do not claim that a high gold word F1 automatically improves downstream classification or that 18 synthetic sentences establish performance on real Persian data.

> TODO: your comparison table, evidence, interpretation, and limitations here.

In [ ]:
# Exercise 5 — inspect examples visually and WRITE your explanation in a Markdown cell.
# The first method is the given gold annotation, not a prediction.
# TODO: Explain at least three cases where the baselines disagree with gold.
for sample_id, text, gold in gold_examples[:3]:
    show_case(text, [
        ('GOLD — reference words', gold),
        ('SPACE — baseline', tokenize_space_to_space(text)),
        ('\\b — naive regex baseline', tokenize_word_boundary(text)),
        ('PERSIAN-AWARE — your implementation', tokenize_persian(text)),
    ], heading=f'Exercise 5 — {sample_id}',
       note='Word methods have comparable span boundaries; BPE has a different unit.')
    pieces = encode_byte_bpe(text, bpe_merges)
    show_bpe_pieces(pieces, title=f'{sample_id} — BPE subword pieces (hex)')
    assert b''.join(pieces).decode('utf-8') == text


---
# 5. Exit ticket

Write concise responses:

1. What is the distinction between `text.split(' ')` and `text.split()`? Give a Persian example with two spaces or a tab.
2. In Python `re`, what does `\b` mean? Why does `\w` not include the ZWNJ in `می‌روم`?
3. What are the **initial units** in our byte-level BPE model, and what decides the next pair to merge?
4. Why may a BPE `bytes` token fail to decode to a visible Unicode character even though concatenating all BPE tokens always restores the full string?
5. Why is word-level span F1 not a fair direct score for byte-level BPE? Name two BPE diagnostics suitable for this exercise.

> TODO: write your answers here.

# Final submission checklist

- [ ] Student metadata and Exercises 1–5 completed.
- [ ] Persian-aware `tokenize_persian` passes **all public checks**.
- [ ] BPE `merge_adjacent_pair` passes **all public checks**.
- [ ] BPE is trained **only** on `data/bpe_train.txt`; gold TSV is used for evaluation only.
- [ ] Three word-level baselines compared via **word-span** precision/recall/F1.
- [ ] BPE compared **separately** via exact round trip, counts and average bytes/token.
- [ ] At least three Persian examples and limitations discussed.
- [ ] Notebook executes via **Kernel → Restart Kernel and Run All Cells**.
- [ ] Save as `STUDENT_ID_session03.ipynb` and submit privately, not on GitHub.

**External help/sources used:**  
> TODO: cite any outside code, ideas, or AI help according to `docs/AI_USE_POLICY_FA.md`.

**End of Session 03.**